# Lesson 4: Enzyme kinetics

**You will:** measure how fast an enzyme works at different substrate amounts, and recover the Michaelis–Menten law, `v = Vmax S / (Km + S)`, from the elementary reactions.

**The model** ([`enzyme.bngl`](enzyme.bngl)): `E + S <-> ES -> E + P`. The enzyme binds the substrate (`k1`), lets go (`km1`) or turns it into product (`k2`). One enzyme molecule, hundreds of substrates.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import bngsim

In [ ]:
model = bngsim.Model.load("enzyme.bngl")
sim = bngsim.Simulator(model, method="ode")
r = sim.run(t_span=(0, 3000), n_points=301)
plt.plot(r.time, r.observables["S"], label="substrate S")
plt.plot(r.time, r.observables["P"], label="product P")
plt.xlabel("time (s)")
plt.ylabel("molecules")
plt.legend()
plt.show()

## The rate at each substrate amount

The product is made at rate `v = k2 · ES`. The enzyme–substrate complex settles within seconds, long before much substrate is used, so we can read the rate at t = 10 s for a range of starting amounts.

Michaelis and Menten predicted that rate from the rate constants alone: `Km = (km1 + k2) / k1` and `Vmax = k2 · E0`.

In [ ]:
k1, km1, k2, E0 = 0.01, 0.4, 0.1, 1
Km, Vmax = (km1 + k2) / k1, k2 * E0
amounts, rates = [], []
for S0 in [5, 10, 25, 50, 100, 200, 500, 1000, 2000]:
    model.set_param("S0", S0)
    model.reset()
    r = sim.run(t_span=(0, 10), n_points=11)
    amounts.append(r.observables["S"][-1])
    rates.append(k2 * r.observables["ES"][-1])
amounts, rates = np.array(amounts), np.array(rates)

S = np.linspace(0, amounts.max(), 200)
plt.plot(amounts, rates, "o", label="simulated")
plt.plot(S, Vmax * S / (Km + S), label="Vmax S / (Km + S)")
plt.axhline(Vmax, color="gray", linestyle=":", label="Vmax")
plt.axvline(Km, color="gray", linestyle="--", label="Km")
plt.xlabel("substrate molecules")
plt.ylabel("product made per second")
plt.legend()
plt.show()

In [ ]:
for s, v in zip(amounts[::2], rates[::2]):
    print(f"S = {s:7.1f}: rate by Michaelis-Menten {Vmax * s / (Km + s):.5f}, simulated {v:.5f}")

## Reading the curve

The law is an approximation (it assumes the complex has settled and the enzyme is scarce), so the two agree to about 0.1% here, not exactly. At low substrate the rate rises in proportion to S; at high substrate the enzyme is always busy and the rate levels off at `Vmax`. Half of `Vmax` is reached at `S = Km`.

## Try it

1. Double `E0`. What happens to `Vmax`, and to `Km`?
2. Make `km1` 100 times smaller, so the enzyme almost never lets go of its substrate. How does `Km` change?
3. Set `E0` to 200, as much enzyme as substrate. Does the Michaelis–Menten law still fit? (It assumes the enzyme is scarce.)